In [ ]:
# Drive mount
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/YOLOComparison'

models_config = {
    'yolov8n':  f'{BASE}/yolov8/weights/best.pt',
    'yolov9c':  f'{BASE}/yolov9/weights/best.pt',
    'yolov10m': f'{BASE}/yolov10/weights/best.pt',
}

for name, path in models_config.items():
    if os.path.exists(path):
        print(f"✅ {name} — best.pt found!")
    else:
        print(f"❌ {name} — best.pt NOT found at {path}")

Mounted at /content/drive
✅ yolov8n — best.pt found!
✅ yolov9c — best.pt found!
✅ yolov10m — best.pt found!


In [ ]:
# STEP 2 — Install & Download Dataset
!pip install roboflow ultralytics -q

from roboflow import Roboflow
rf = Roboflow(api_key="MW5HE7ZZxqr3eOxhqYbq")
project = rf.workspace("free-roboflow").project("ktproject")
version = project.version(11)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 61.2 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to KTproject-11 in yolov8:: 100%|██████████| 17915/17915 [00:02<00:00, 6317.56it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# STEP 3 — Test Split Recreate
import os, random, shutil

train_images = "/content/KTproject-11/train/images"
train_labels = "/content/KTproject-11/train/labels"
test_images  = "/content/KTproject-11/test/images"
test_labels  = "/content/KTproject-11/test/labels"

os.makedirs(test_images, exist_ok=True)
os.makedirs(test_labels, exist_ok=True)

test_ratio  = 0.15
image_files = [f for f in os.listdir(train_images) if f.endswith(('.jpg','.png','.jpeg'))]
random.seed(42)  # same split hoga har baar
random.shuffle(image_files)
test_files  = image_files[:int(len(image_files) * test_ratio)]

for img in test_files:
    label = img.rsplit(".", 1)[0] + ".txt"
    src_img = os.path.join(train_images, img)
    src_lbl = os.path.join(train_labels, label)
    if os.path.exists(src_img) and os.path.exists(src_lbl):
        shutil.move(src_img, os.path.join(test_images, img))
        shutil.move(src_lbl, os.path.join(test_labels, label))

print(f"✅ Test split ready! Total test images: {len(os.listdir(test_images))}")

✅ Test split ready! Total test images: 1245


In [ ]:
# STEP 4 — data.yaml mein test path add karo
with open('/content/KTproject-11/data.yaml', 'r') as f:
    content = f.read()

print("Current yaml:")
print(content)

if 'test:' not in content:
    content = content.replace(
        'train: train/images',
        'train: train/images\ntest: test/images'
    )
    with open('/content/KTproject-11/data.yaml', 'w') as f:
        f.write(content)
    print("\n✅ test path added!")
else:
    print("\n✅ test path already exists!")

Current yaml:
names:
- Landslides
nc: 1
roboflow:
  license: CC BY 4.0
  project: ktproject
  url: https://universe.roboflow.com/free-roboflow/ktproject/dataset/11
  version: 11
  workspace: free-roboflow
test: /content/KTproject-11/test/images../test/images
train: ../train/images
val: ../valid/images


✅ test path already exists!


In [ ]:
# STEP 5 — TEST EVALUATION
!pip install ultralytics -q

from ultralytics import YOLO
import pandas as pd

BASE_WEIGHTS = '/content/drive/MyDrive/YOLOComparison'
DATA_YAML    = '/content/KTproject-11/data.yaml'
SAVE_PATH    = '/content/drive/MyDrive/YOLOComparison/test_results'

models_config = {
    'yolov8n':  f'{BASE_WEIGHTS}/yolov8/weights/best.pt',
    'yolov9c':  f'{BASE_WEIGHTS}/yolov9/weights/best.pt',
    'yolov10m': f'{BASE_WEIGHTS}/yolov10/weights/best.pt',
}

test_results = {}

for name, weights_path in models_config.items():
    print(f"\n{'='*50}")
    print(f"🔍 Evaluating {name} on TEST SET...")
    print(f"{'='*50}")

    model = YOLO(weights_path)

    results = model.val(
        data=DATA_YAML,
        split='test',
        imgsz=640,
        batch=16,
        plots=True,
        project=SAVE_PATH,
        name=f'{name}_test',
    )

    test_results[name] = {
        'mAP50':     round(results.box.map50, 4),
        'mAP50_95':  round(results.box.map,   4),
        'Precision': round(results.box.mp,    4),
        'Recall':    round(results.box.mr,    4),
    }
    test_results[name]['F1'] = round(
        2 * test_results[name]['Precision'] * test_results[name]['Recall'] /
        (test_results[name]['Precision'] + test_results[name]['Recall'] + 1e-8), 4
    )
    print(f"✅ {name} done — mAP50: {test_results[name]['mAP50']}")

# FINAL SUMMARY
df = pd.DataFrame(test_results).T
print("\n" + "="*60)
print("     🏆 TEST SET RESULTS — LANDSLIDE DETECTION")
print("="*60)
print(df.to_string())
print("="*60)

for metric in ['mAP50', 'mAP50_95', 'Precision', 'Recall', 'F1']:
    best = df[metric].idxmax()
    val  = df[metric].max()
    print(f"🥇 Best {metric:12s} → {best.upper()} ({val:.4f})")

print("="*60)
df.to_csv(f'{SAVE_PATH}/test_results_summary.csv')
print(f"\n✅ Results saved to Drive!")


🔍 Evaluating yolov8n on TEST SET...
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1810.3±504.5 MB/s, size: 53.4 KB)
val: Scanning /content/KTproject-11/test/labels... 1245 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1245/1245 1.9Kit/s 0.7s
val: New cache created: /content/KTproject-11/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 78/78 5.5it/s 14.1s
                   all       1245       2813      0.873      0.787      0.889      0.649
Speed: 1.4ms preprocess, 3.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /content/drive/MyDrive/YOLOComparison/test_results/yolov8n_test3
✅ yolov8n done — mAP50: 0.8887

🔍 Evaluating yolov9c on TEST SET...
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4